# Pl@ntNet-300K — top-100 species, ResNet-50, export to ONNX

A second dataset for the same project: instead of the studio shots of
Flowers-102, these are photos from the Pl@ntNet app, taken in the field on
phones. The output is the same pair of files for `model/`:
`plantnet100.onnx` and `plantnet100.labels.json`.

### Why the top 100 species and not all 1,081

The full dataset holds 306,146 images with a long tail: a handful of species
take up most of the photos while hundreds are represented by a dozen each. On
the full set ResNet-50 lands somewhere around 80–90% top-1, and that is the
ceiling of the dataset itself rather than a question of training time. We take
the hundred best-represented species from the slice we download and cap how
many photos each contributes — the classes even out and training fits into a
single Colab session.

### Why the Hugging Face mirror and not Zenodo

The original on Zenodo is a single 31.7 GB zip served slowly: measured from
Colab it gives 0.7 MB/s against 11 MB/s from Hugging Face — hours against
minutes.

Speed is not the whole story though. On Hugging Face the dataset is split into
51 parquet files **ordered by species**: each file holds a contiguous run of
species in full. So there is no need to download everything — the first
thirteen files (≈6.5 GB) cover roughly 270 species with all of their photos,
and we only need a hundred of those.

### How long it takes

| Step | Time |
|---|---|
| Download 13 parquet files (≈6.5 GB) | ~5 min |
| Build the subset, re-encoding images | ~3 min |
| Training, 6 + 12 epochs | ~30 min |

The subset is cached on Google Drive (about 0.7 GB). On a later run the
notebook finds it and skips the download.

Our run reached 79.2% top-1 at 55.5% macro-average: the gap between the two is
exactly the long-tail effect this dataset was built to expose.

### Before you start

`Runtime` → `Change runtime type` → **GPU**. Then `Run all`.

## 0. Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU FOUND: Runtime -> Change runtime type -> GPU"
!pip install -q onnx onnxruntime

In [ ]:
import io, json, os, shutil, tarfile, time, random
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torchvision
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet50, ResNet50_Weights

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | device {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: training on a CPU takes many hours. Enable the GPU.")

## 1. Settings

`MAX_TRAIN_PER_SPECIES` is the main lever. It also fixes the class imbalance:
without a cap the most common species would contribute hundreds of times more
examples than the hundredth one.

In [ ]:
NUM_SPECIES = 100              # how many species to keep (by photo count)
MAX_TRAIN_PER_SPECIES = 500    # cap on training photos per species
MAX_EVAL_PER_SPECIES = 100     # cap per species in val/test, to keep evaluation quick
TRAIN_SHARDS = 13              # how many parquet files to download (of 51)

IMG_SIZE = 224
RESIZE_SIZE = 256
STORED_SIZE = 256              # short side the images are re-encoded to

HF_REPO = "mikehemberger/plantnet300K"
# Scientific species names: the Pl@ntNet repository no longer carries this
# file, but the mirror keeps it next to the data.
SPECIES_META = "data/train/metadata.csv"

WORK = Path("/content")
SHARD_DIR = WORK / "parquet"
SUBSET_DIR = WORK / "plantnet_subset"
CACHE_NAME = f"plantnet_top{NUM_SPECIES}_{MAX_TRAIN_PER_SPECIES}.tar"

## 2. Google Drive as a cache

Optional but recommended: only the assembled subset goes to Drive (about
0.7 GB), not the downloaded parquet files. Without Drive, a dropped session
means downloading the data again.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_PATH = Path("/content/drive/MyDrive") / CACHE_NAME
    print(f"subset cache: {CACHE_PATH}")
except Exception as exc:
    CACHE_PATH = WORK / CACHE_NAME
    print(f"Drive unavailable ({exc}), caching on the session disk only")

HAVE_CACHE = CACHE_PATH.exists()
print("cache found, no download needed" if HAVE_CACHE
      else "no cache — downloading parquet files from Hugging Face")

## 3. Downloading the parquet files

Only the first `TRAIN_SHARDS` files of the training split are fetched. The val
and test files are taken one at a time until they cover the range of labels
that ended up in training: every split is ordered by species the same way, so
nothing extra comes down.

`hf_hub_download` resumes interrupted transfers and caches them — rerunning
the cell after a drop continues where it stopped.

In [ ]:
# File names in the mirror do not match the split names: validation lives
# under validation-*, not val-*. So read the repository listing instead of
# guessing prefixes.
SPLIT_PREFIX = {"train": "train", "val": "validation", "test": "test"}

if not HAVE_CACHE:
    available = HfApi().list_repo_files(HF_REPO, repo_type="dataset")
    SHARDS = {}
    for split, prefix in SPLIT_PREFIX.items():
        SHARDS[split] = sorted(
            f for f in available
            if f.startswith(f"data/{prefix}-") and f.endswith(".parquet")
        )
        assert SHARDS[split], (
            f"no files for the {split} split (looked for data/{prefix}-*.parquet). "
            f"Similar paths: {[f for f in available if prefix in f][:3]}"
        )
        print(f"{split}: {len(SHARDS[split])} files")

    def fetch(path):
        return hf_hub_download(HF_REPO, path, repo_type="dataset",
                               local_dir=str(SHARD_DIR))

    started = time.time()
    with ThreadPoolExecutor(max_workers=4) as pool:
        train_files = list(pool.map(fetch, SHARDS["train"][:TRAIN_SHARDS]))
    volume = sum(Path(f).stat().st_size for f in train_files)
    print(f"\ndownloaded {volume / 1e9:.1f} GB in {(time.time() - started) / 60:.1f} min")

## 4. Choosing the species

The class names sit right in the parquet metadata — Pl@ntNet species ids,
1,081 of them, in the same order as the numeric labels.

The last species in the last downloaded file is dropped: it is almost
certainly cut by the file boundary, so its photos would be counted
incompletely.

In [ ]:
if not HAVE_CACHE:
    schema_meta = pq.ParquetFile(train_files[0]).schema_arrow.metadata
    hf_info = json.loads(schema_meta[b"huggingface"].decode())
    SPECIES_IDS = hf_info["info"]["features"]["label"]["names"]
    print(f"species in the dataset: {len(SPECIES_IDS)}")

    counts = Counter()
    for path in train_files:
        counts.update(pq.read_table(path, columns=["label"]).column("label").to_pylist())

    truncated = max(counts)          # the species cut by the file boundary
    del counts[truncated]
    print(f"species fully contained in the downloaded slice: {len(counts)}")

    CHOSEN = [label for label, _ in counts.most_common(NUM_SPECIES)]
    CHOSEN_SET = set(CHOSEN)
    print(f"selected: {len(CHOSEN)} species")
    print(f"photos for the most common one: {counts[CHOSEN[0]]:,}")
    print(f"photos for the hundredth:       {counts[CHOSEN[-1]]:,}")
    kept = sum(min(counts[l], MAX_TRAIN_PER_SPECIES) for l in CHOSEN)
    print(f"capped at {MAX_TRAIN_PER_SPECIES} per species, {kept:,} photos remain")

## 5. Assembling the subset

The chosen species are unpacked into the plain folder layout `ImageFolder`
understands, shrinking every image to a 256 px short side along the way.
Training in Colab is bottlenecked by JPEG decoding on two cores rather than by
the GPU, so resizing once pays for itself on every epoch.

In [ ]:
def shrink(raw):
    image = Image.open(io.BytesIO(raw)).convert("RGB")
    width, height = image.size
    scale = STORED_SIZE / min(width, height)
    if scale < 1:
        image = image.resize((max(1, round(width * scale)), max(1, round(height * scale))),
                             Image.BILINEAR)
    return image


def write_split(paths, split, limit):
    kept = Counter()
    pool = ThreadPoolExecutor(max_workers=os.cpu_count() * 2)
    futures = []

    def save(raw, dest):
        shrink(raw).save(dest, "JPEG", quality=90)

    for path in paths:
        parquet = pq.ParquetFile(path)
        for batch in parquet.iter_batches(batch_size=256, columns=["image", "label"]):
            labels = batch.column("label").to_pylist()
            images = batch.column("image").to_pylist()
            for image, label in zip(images, labels):
                if label not in CHOSEN_SET or kept[label] >= limit:
                    continue
                out_dir = SUBSET_DIR / split / SPECIES_IDS[label]
                out_dir.mkdir(parents=True, exist_ok=True)
                futures.append(pool.submit(save, image["bytes"], out_dir / f"{kept[label]:04d}.jpg"))
                kept[label] += 1

    pool.shutdown(wait=True)
    failed = sum(1 for f in futures if f.exception())
    print(f"{split}: {sum(kept.values()):,} photos, {len(kept)} species"
          + (f", {failed} failed" if failed else ""))

    # An empty split means the filters are wrong, not an acceptable outcome.
    # Fail here, while the downloaded files are still around.
    assert kept, f"not a single photo landed in the {split} split"
    return sum(kept.values())


def collect_eval(split):
    """Fetch the split's files one by one until the needed labels are covered."""
    paths, ceiling = [], max(CHOSEN)
    for remote in SHARDS[split]:
        local = fetch(remote)
        paths.append(local)
        seen = pq.read_table(local, columns=["label"]).column("label").to_pylist()
        if max(seen) > ceiling:
            break
    return paths


if not HAVE_CACHE:
    started = time.time()
    write_split(train_files, "train", MAX_TRAIN_PER_SPECIES)
    for split in ("val", "test"):
        write_split(collect_eval(split), split, MAX_EVAL_PER_SPECIES)
    print(f"subset assembled in {(time.time() - started) / 60:.1f} min")

    with tarfile.open(CACHE_PATH, "w") as tar:
        tar.add(SUBSET_DIR, arcname=SUBSET_DIR.name)
    print(f"cache saved: {CACHE_PATH.stat().st_size / 1e9:.2f} GB")

    shutil.rmtree(SHARD_DIR, ignore_errors=True)   # the parquet files are no longer needed
else:
    with tarfile.open(CACHE_PATH) as tar:
        tar.extractall(WORK)
    print("subset restored from the cache")

for split in ("train", "val", "test"):
    folders = sorted((SUBSET_DIR / split).iterdir())
    images = list((SUBSET_DIR / split).rglob("*.jpg"))
    print(f"{split}: {len(images):,} photos, {len(folders)} species")

## 6. Dataset

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(RESIZE_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_set = ImageFolder(SUBSET_DIR / "train", transform=train_tf)
val_set = ImageFolder(SUBSET_DIR / "val", transform=eval_tf)
test_set = ImageFolder(SUBSET_DIR / "test", transform=eval_tf)

assert train_set.classes == val_set.classes == test_set.classes, "splits disagree on classes"
NUM_CLASSES = len(train_set.classes)
print(f"classes: {NUM_CLASSES} | train {len(train_set):,} | val {len(val_set):,} | test {len(test_set):,}")

BATCH_SIZE = 64
NUM_WORKERS = os.cpu_count()

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          drop_last=True, persistent_workers=True)
val_loader = DataLoader(val_set, batch_size=128, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

The folders are named by numeric species id, which is not something to show in
a chat. The scientific names come from `metadata.csv` next to the data: 66 MB
covering all 1,081 species.

In [ ]:
import csv

meta_path = hf_hub_download(HF_REPO, SPECIES_META, repo_type="dataset",
                            local_dir=str(WORK / "meta"))

species_id_2_name = {}
with open(meta_path, newline="") as handle:
    for row in csv.DictReader(handle):
        species_id_2_name.setdefault(row["class"], row["species"])

# ImageFolder sorts the folders and that order defines the label indices —
# the names have to follow exactly the same order.
CLASS_NAMES = [species_id_2_name.get(sp, f"species {sp}") for sp in train_set.classes]
missing = sum(1 for n in CLASS_NAMES if n.startswith("species "))
print(f"names resolved: {len(CLASS_NAMES) - missing} of {len(CLASS_NAMES)}")
print("first five:", CLASS_NAMES[:5])

In [ ]:
import matplotlib.pyplot as plt

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax in axes.flat:
    img, label = train_set[random.randrange(len(train_set))]
    ax.imshow((img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
    ax.set_title(CLASS_NAMES[label], fontsize=7)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Model and training

In [ ]:
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

def set_backbone_trainable(model, names):
    for param in model.parameters():
        param.requires_grad = False
    for name in list(names) + ["fc"]:
        for param in getattr(model, name).parameters():
            param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"trainable parameters: {trainable:,} of {total:,} ({100 * trainable / total:.1f}%)")

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


@torch.no_grad()
def evaluate(model, loader):
    """Return top-1 and macro-average top-1 — the mean across species.

    On long-tailed data the two diverge: plain accuracy rides on the common
    species, while the macro figure shows what happens to the rare ones.
    """
    model.eval()
    correct = torch.zeros(NUM_CLASSES)
    total = torch.zeros(NUM_CLASSES)
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        preds = model(images).argmax(dim=1).cpu()
        for label, pred in zip(labels, preds):
            total[label] += 1
            correct[label] += int(label == pred)
    seen = total > 0
    return (correct.sum() / total.sum()).item(), (correct[seen] / total[seen]).mean().item()


def train_phase(model, epochs, lr, trainable_blocks, label):
    set_backbone_trainable(model, trainable_blocks)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=len(train_loader))
    scaler = torch.amp.GradScaler(DEVICE, enabled=(DEVICE == "cuda"))

    for epoch in range(1, epochs + 1):
        model.train()
        started = time.time()
        running_loss = seen = 0
        for images, labels in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE, enabled=(DEVICE == "cuda")):
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item() * labels.numel()
            seen += labels.numel()
        print(f"[{label}] epoch {epoch}/{epochs}  loss {running_loss / seen:.4f}"
              f"  {time.time() - started:.0f} s")
    return model

### Phase 1 — the classifier only

In [ ]:
train_phase(model, epochs=6, lr=1e-3, trainable_blocks=[], label="phase 1")
acc1, macro1 = evaluate(model, val_loader)
print(f"\nval after phase 1: top-1 {acc1:.4f} | macro {macro1:.4f}")

### Phase 2 — unfreezing layer3 and layer4

In [ ]:
train_phase(model, epochs=12, lr=1e-4, trainable_blocks=["layer3", "layer4"], label="phase 2")
acc2, macro2 = evaluate(model, val_loader)
print(f"\nval after phase 1: top-1 {acc1:.4f} | macro {macro1:.4f}")
print(f"val after phase 2: top-1 {acc2:.4f} | macro {macro2:.4f}")

In [ ]:
acc_test, macro_test = evaluate(model, test_loader)
print(f"TEST: top-1 {acc_test:.4f} | macro-average {macro_test:.4f}")

torch.save(model.state_dict(), "resnet50_plantnet100.pt")
print("weights saved: resnet50_plantnet100.pt")

## 8. Exporting to ONNX

Exactly the same contract as the Flowers-102 model: normalisation and softmax
inside the graph, input `1x3x224x224` in `[0,1]`. The server picks the new
model up without a single code change.

In [ ]:
class ExportWrapper(nn.Module):
    """Takes RGB in [0,1], normalises inside and returns probabilities."""

    def __init__(self, model, mean, std):
        super().__init__()
        self.model = model
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, pixel_values):
        return torch.softmax(self.model((pixel_values - self.mean) / self.std), dim=1)


wrapper = ExportWrapper(model, IMAGENET_MEAN, IMAGENET_STD).to("cpu").eval()
dummy = torch.rand(1, 3, IMG_SIZE, IMG_SIZE)

ONNX_PATH = "plantnet100.onnx"
export_kwargs = dict(
    input_names=["pixel_values"],
    output_names=["probabilities"],
    opset_version=17,
    dynamic_axes={"pixel_values": {0: "batch"}, "probabilities": {0: "batch"}},
)
try:
    torch.onnx.export(wrapper, dummy, ONNX_PATH, dynamo=False, **export_kwargs)
except TypeError:
    torch.onnx.export(wrapper, dummy, ONNX_PATH, **export_kwargs)

print(f"{ONNX_PATH}: {Path(ONNX_PATH).stat().st_size / 1e6:.1f} MB")

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

# Take random test photos rather than the first batch: it is sorted by class.
indices = random.sample(range(len(test_set)), 8)
images = torch.stack([test_set[i][0] for i in indices])
labels = [test_set[i][1] for i in indices]
batch = (images * std + mean).clamp(0, 1)

with torch.no_grad():
    torch_probs = wrapper(batch).numpy()
onnx_probs = session.run(None, {"pixel_values": batch.numpy()})[0]

max_diff = np.abs(torch_probs - onnx_probs).max()
print(f"largest probability difference: {max_diff:.2e}")
assert max_diff < 1e-4, "torch and onnxruntime disagree — the export is broken"

print("\nonnxruntime predictions:")
for i, true_label in enumerate(labels):
    top = int(onnx_probs[i].argmax())
    mark = "+" if top == true_label else "-"
    print(f"  {mark} {CLASS_NAMES[top][:34]:<34} {onnx_probs[i][top]:.3f}"
          f"   (truth: {CLASS_NAMES[true_label][:30]})")

## 9. Metadata and download

In [ ]:
meta = {
    "architecture": "resnet50",
    "dataset": f"PlantNet-{NUM_SPECIES}",
    "num_classes": NUM_CLASSES,
    "classes": CLASS_NAMES,
    "accuracy": {
        "test_top1_head_only": round(acc1, 4),
        "test_top1_finetuned": round(acc_test, 4),
        "test_macro_top1": round(macro_test, 4),
        "test_size": len(test_set),
    },
    "preprocessing": {
        "resize_shorter_side": RESIZE_SIZE,
        "center_crop": IMG_SIZE,
        "scale": "divide by 255, range [0,1]",
        "normalization": "inside the ONNX graph",
        "input_name": "pixel_values",
        "input_shape": [1, 3, IMG_SIZE, IMG_SIZE],
        "output_name": "probabilities",
        "output": "softmax probabilities",
    },
}

Path("plantnet100.labels.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print(json.dumps({k: v for k, v in meta.items() if k != "classes"}, indent=2, ensure_ascii=False))

In [ ]:
from google.colab import files

files.download("plantnet100.onnx")
files.download("plantnet100.labels.json")

Both files go into the project's `model/` folder under those same names. The
server reads the class count from the graph and the names and metrics from the
description beside it, so switching between the Flowers-102 and Pl@ntNet
models is a matter of replacing two files and restarting `uvicorn`.

Worth re-checking the "not sure" threshold in `app/inference.py`: it was tuned
for the 102 classes of Flowers-102.